# Technologies.csv Generation for Norte Amazonica 2025

Generates one `Technologies.csv` per cluster (C1–C5) for EnergyScope ESMC.
Regional file carries capacity parameters only (`c_p`, `fmin_perc`, `fmax_perc`, `f_min`, `f_max`); cost parameters stay in `00_INDEP/Technologies.csv`.
Base: SA-PA reference file (Pablo Jimenez Zabalaga, 2025). Only values documented below are overwritten.

In [1]:
import os
import unicodedata
import pandas as pd

OUT_DIR = "output_energyscope"

CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir",
        "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando",
        "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}

# SA-PA reference file — Pablo Jimenez Zabalaga, 2025
base_df = pd.read_csv("../data/Technologies.csv", sep=";")
base_df["Technologies param"] = base_df["Technologies param"].str.strip()

# Structural columns to match the EnergyScope model's Technologies.csv (f_min_prod/f_max_prod
# are absolute annual production floor/ceiling, GWh/y — inactive at these defaults)
base_df["f_min_prod"] = 0.0
base_df["f_max_prod"] = 1e15

# Total households per municipality — Bolivia Census 2024
pop_df = pd.read_csv("../../exctraction of data/output/CSV_final.csv")
HH_COL = "NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Total"

def normalize(s):
    """Strip accents + lowercase + spaces→underscore, for name matching."""
    s = unicodedata.normalize("NFD", str(s))
    return "".join(c for c in s if unicodedata.category(c) != "Mn").lower().replace(" ", "_")

hh_lookup = {
    normalize(row["MUNICIPIO/TIOC"]): row[HH_COL]
    for _, row in pop_df.dropna(subset=["MUNICIPIO/TIOC", HH_COL]).iterrows()
}

cluster_pop = {}
for k, munis in CLUSTERS.items():
    cluster_pop[k] = sum(hh_lookup.get(normalize(m), 0) for m in munis)

total_pop = sum(cluster_pop.values())

for k, v in cluster_pop.items():
    print(f"C{k}: {v:,.0f} hh")
print(f"Total: {total_pop:,.0f} hh")

C1: 8,178 hh
C2: 802 hh
C3: 40,298 hh
C4: 15,752 hh
C5: 15,564 hh
Total: 80,594 hh


## 1. GENSET_DIESEL

`f_min` = existing installed diesel capacity (cannot be decommissioned); `f_max = 1e15`.
Values from AETN Bolivia Anuario Estadístico 2024, Cuadro VI-1 (*Potencia Instalada Térmica en Sistemas Aislados*) — **active power (MW), not apparent power (MVA)**.
Municipalities without AETN data (non-electrified or below threshold) get `f_min = 0`.
C1 is SIN-connected via ENDE DELBENI — no local diesel generation.

In [2]:
# Source: AETN Bolivia Anuario Estadístico 2024, Cuadro VI-1 — Potencia Instalada Térmica (MW activos)
GENSET_AETN_MW = {
    "Riberalta":              28.00,  # ENDE isolated system
    "Guayaramerín":           22.05,  # ENDE DELBENI "Guayaramerín Unificado" — MW activos, Cuadro VI-1
    "Puerto_Gonzalo_Moreno":   3.28,  # AETN 2024 Cuadro VI-1 — MW activos
    "Sena":                    5.10,  # "El Sena" AETN 2024 Cuadro VI-1 — MW activos
    "Villa_Nueva":             2.32,  # AETN 2024 Cuadro VI-1 — MW activos
    "Cobija":                 27.36,  # ENDE isolated system — MW activos, Cuadro VI-1
    # SIN-connected → no local diesel generation
    "Exaltación":              0.00,  # SIN via San Ignacio de Moxos
    "Reyes":                   0.00,  # SIN (ENDE DELBENI)
    "Santa_Rosa_Beni":         0.00,  # SIN (ENDE DELBENI)
    "Ixiamas":                 0.00,  # SIN-connected, no AETN isolated data
    # No AETN data → non-electrified or below AETN threshold
    "Bolpebra": 0.0, "Bella_Flor": 0.0, "Filadelfia": 0.0, "Ingavi": 0.0,
    "Nueva_Esperanza": 0.0, "Porvenir": 0.0, "Puerto_Rico": 0.0,
    "San_Lorenzo": 0.0, "San_Pedro": 0.0, "Santa_Rosa_Pando": 0.0,
    "Santos_Mercado": 0.0,
}

# Sum per cluster in GW
genset_fmin_gw = {}
for k, munis in CLUSTERS.items():
    genset_fmin_gw[k] = sum(GENSET_AETN_MW.get(m, 0.0) for m in munis) / 1000.0
    print(f"C{k}: {genset_fmin_gw[k]:.5f} GW")

C1: 0.00000 GW
C2: 0.00000 GW
C3: 0.05333 GW
C4: 0.00742 GW
C5: 0.02736 GW


## 2. Cooking stoves

`f_min` is derived from current cooking practices (Census 2024) so the optimizer cannot remove existing stove capacity:

$$f_{\min} = \frac{n_{\text{hh}} \times FE}{\text{COEFF} \times 10^6 \times c_p \times 8760}$$

Energy intensities from a rural Bolivia field study (Oriente/Yungas zones). EnergyScope coefficients from `Layers_in_out.csv`.

In [3]:
# Source: Bolivia Census 2024, CSV_final — (hh_wood, hh_lpg, hh_elec)
COOKING_DATA = {
    "Exaltación":            (1090, 332,   1),
    "Reyes":                 (1482, 1848,  4),
    "Santa_Rosa_Beni":       (917,  1764,  6),
    "Ixiamas":               (1528, 1679,  1),
    "Bolpebra":              (340,  452,   1),
    "Guayaramerín":          (1575, 8506, 19),
    "Riberalta":             (4512, 20740, 47),
    "Puerto_Gonzalo_Moreno": (927,  991,   3),
    "Bella_Flor":            (409,  803,   2),
    "Filadelfia":            (613,  1857,  3),
    "Ingavi":                (350,  293,   0),
    "Nueva_Esperanza":       (190,  293,   2),
    "Porvenir":              (223,  1950,  2),
    "Puerto_Rico":           (405,  1513,  6),
    "San_Lorenzo":           (763,  1050,  1),
    "San_Pedro":             (520,  85,    0),
    "Santa_Rosa_Pando":      (235,  615,   2),
    "Santos_Mercado":        (359,  238,   0),
    "Sena":                  (1312, 1487,  2),
    "Villa_Nueva":           (396,  302,   1),
    "Cobija":                (300,  13727, 70),
}

FE_WOOD = 6760  # kWh/hh/yr — Source: Perplexity rural Bolivia field study (Oriente/Yungas)
FE_LPG  = 6970  # kWh/hh/yr — same source

COEFF_WOOD = 6.25      # Source: Layers_in_out.csv — STOVE_WOOD, COOKING column
COEFF_LPG  = 1.559258  # Source: Layers_in_out.csv — STOVE_LPG,  COOKING column
CP_STOVE   = 0.1875    # c_p for all stoves (SA-PA base)

stove_wood_fmin = {}
stove_lpg_fmin  = {}

for k, munis in CLUSTERS.items():
    wood_gwh = 0
    lpg_gwh  = 0
    for m in munis:
        hh_wood, hh_lpg, _ = COOKING_DATA[m]
        wood_gwh += hh_wood * FE_WOOD / COEFF_WOOD / 1e6
        lpg_gwh  += hh_lpg  * FE_LPG  / COEFF_LPG  / 1e6
    stove_wood_fmin[k] = wood_gwh / (CP_STOVE * 8760)
    stove_lpg_fmin[k]  = lpg_gwh  / (CP_STOVE * 8760)

print(f"{'':8} {'STOVE_WOOD':>12} {'STOVE_LPG':>12}")
for k in range(1, 6):
    print(f"C{k}       {stove_wood_fmin[k]:12.7f} {stove_lpg_fmin[k]:12.7f}")

           STOVE_WOOD    STOVE_LPG
C1          0.0033037    0.0153030
C2          0.0002239    0.0012301
C3          0.0046188    0.0822902
C4          0.0038029    0.0285377
C5          0.0001976    0.0373581


## 3. PV_UTILITY and population-scaled technologies

In [4]:
# Source: AETN 2024 — Cobija only: 5.10 MW (ENDE GUARACACHI S.A.)
PV_FMIN_GW = {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.00510}

# Technologies whose SA-PA f_min is scaled by the cluster's share of total households.
# Not in this list: GENSET_DIESEL, PV_UTILITY, STOVE_* (computed above),
# REGASIFICATION, LNG_STORAGE (see section 4).
# Infrastructure (HVAC_LINE, GAS_PIPELINE …) and LED fmin_perc are left unchanged.
POPULATION_SCALED = [
    "DIESEL_STORAGE", "GASOLINE_STORAGE", "LPG_STORAGE",
    "JET_FUEL_STORAGE",
    "CAR_GASOLINE_PRIVATE", "SUV_GASOLINE_PRIVATE",
    "PICKUP_TRUCK_GASOLINE_PRIVATE", "MOTORCYCLE_GASOLINE_PRIVATE",
    "BUS_GASOLINE_PUBLIC", "CAR_DIESEL_PRIVATE", "SUV_DIESEL_PRIVATE",
    "PICKUP_TRUCK_DIESEL_PRIVATE", "BUS_DIESEL_PUBLIC",
    "TRUCK_DIESEL_P", "VAN_DIESEL", "TRUCK_GASOLINE", "VAN_GASOLINE",
    "FISH_MACHINERY_DIESEL", "FISH_MACHINERY_EL",
    "IND_MACHINERY_EL", "COMM_MACHINERY_DIESEL", "COMM_MACHINERY_EL",
    "AGR_MACHINERY_DIESEL", "AGR_MACHINERY_EL",
    "DEC_DIRECT_ELEC", "DEC_BOILER_GAS",
    "IND_BOILER_WOOD", "IND_BOILER_OIL", "IND_BOILER_DIESEL",
    "STOVE_NG", "STOVE_OIL",
]

# Pre-read SA-PA f_min for each technology before the cluster loop
base_fmin = {}
for tech in POPULATION_SCALED:
    row = base_df.loc[base_df["Technologies param"] == tech, "f_min"]
    base_fmin[tech] = float(row.values[0]) if len(row) else 0.0

## 4. REGASIFICATION and LNG_STORAGE

`f_min = 0` for both in all clusters — not population-scaled.
AETN 2024 confirms no LNG regasification stations in Norte Amazónica
(Riberalta, Guayaramerín, Cobija all use diesel only for generation).

## 5. SHS rows and output generation

Three Solar Home System technologies are appended to every cluster. Cost parameters are in
`00_INDEP/Technologies.csv` (Roger Arias thesis); `share_dispersion` is calibrated after the
pass-1 solve by `EnergyScope/scripts/bloc30_calibrate.py` and written to `Misc.json`, so it is
not set here.

Two per-cluster quantities come from the live GIS pipeline
(`analyse_GIS_phase2_projections/output/`):

| Column | Source | Meaning |
|---|---|---|
| `f_min` of `PV_HS` / `HS_DIESEL` / `BATT_HS` | `share_dispersion_final_BC.csv` | Brownfield credit: the panels and gensets Source-B dispersed households already own. Any scenario that hands the bundle to B and C inherits them, so the optimizer cannot decommission them. |
| `f_max_prod` of `PV_HS` and `HS_DIESEL` (dual cap) | `2024/cluster_summary.csv`, `demande_dispersee_GWh` | The pair cannot serve more than the annual demand of the households that stay off-grid. |

Both were previously hardcoded to `0` and `1e15` here, which dropped the brownfield credit and
removed the cap — the published catalogue in `Data/2025/sufficiency` has carried the real values
since 2026-08-17. `BATT_HS.f_max_prod` stays uncapped: storage moves energy, it does not produce it.

In [5]:
GIS_OUT = "../../analyse_GIS_phase2_projections/output"

# Brownfield credit on the off-grid fleet already owned by dispersed Source-B households.
share_disp = pd.read_csv(f"{GIS_OUT}/share_dispersion_final_BC.csv").set_index("Cluster")
pv_hs_fmin     = {k: float(share_disp.loc[f"C{k}", "f_min_PV_HS_GW"])     for k in range(1, 6)}
hs_diesel_fmin = {k: float(share_disp.loc[f"C{k}", "f_min_HS_DIESEL_GW"]) for k in range(1, 6)}
batt_hs_fmin   = {k: float(share_disp.loc[f"C{k}", "f_min_BATT_HS_GWh"])  for k in range(1, 6)}

# Dual f_max_prod cap = annual demand of the households that stay off-grid, greedy-tree
# classification (the same quantity bloc30_calibrate.py uses as its target).
cluster_summary = pd.read_csv(f"{GIS_OUT}/2024/cluster_summary.csv").set_index("Cluster")
dispersed_demand_gwh = {k: float(cluster_summary.loc[f"C{k}", "demande_dispersee_GWh"])
                        for k in range(1, 6)}

print(f"{'':6} {'PV_HS f_min':>13} {'HS_DIESEL f_min':>17} {'BATT_HS f_min':>15} {'f_max_prod':>12}")
for k in range(1, 6):
    print(f"C{k}     {pv_hs_fmin[k]:>13.6g} {hs_diesel_fmin[k]:>17.6g} "
          f"{batt_hs_fmin[k]:>15.6g} {dispersed_demand_gwh[k]:>12.4f}")

         PV_HS f_min   HS_DIESEL f_min   BATT_HS f_min   f_max_prod
C1           3.6e-06         1.335e-05        8.85e-06       0.1794
C2                 0                 0               0       0.0000
C3           8.4e-07          5.78e-06        2.07e-06       0.1107
C4           4.9e-07         2.407e-05         1.2e-06       0.2419
C5                 0                 0               0       0.0000


## 5b. LED-only lighting

Household electricity demand for lighting is derived from RAMP at the LED end-use efficiency
(`LIGHTING_R_C`/`LIGHTING_P` → `LED_BULB`/`LED_LIGHT`, `ELECTRICITY` coefficient 2.941176 in
`Layers_in_out.csv`), and RAMP's own appliance model assumes 7 W LED bulbs. Any other technology
mix serving that demand — incandescent/CFL bulbs at a different efficiency — would be inconsistent
with the demand-side data the model was built from. `LED_BULB`/`LED_LIGHT` are therefore forced on
(`fmin_perc = fmax_perc = 1.0`) and `CONVENTIONAL_BULB`/`CONVENTIONAL_LIGHT` off (`f_max = 0`).

In [6]:
def shs_rows_for(k):
    """SHS rows for cluster k: brownfield f_min, dual f_max_prod cap on PV_HS/HS_DIESEL."""
    return [
        # tech        c_p   fmin_perc fmax_perc  f_min              f_max  f_min_prod  f_max_prod
        ("PV_HS",     1.00, 0, 1, pv_hs_fmin[k],     1e15, 0, dispersed_demand_gwh[k]),
        ("HS_DIESEL", 0.85, 0, 1, hs_diesel_fmin[k], 1e15, 0, dispersed_demand_gwh[k]),
        ("BATT_HS",   1.00, 0, 1, batt_hs_fmin[k],   1e15, 0, 1e15),
    ]

for k, munis in CLUSTERS.items():
    df = base_df.copy()
    share = cluster_pop[k] / total_pop

    df.loc[df["Technologies param"] == "GENSET_DIESEL", "f_min"] = genset_fmin_gw[k]
    df.loc[df["Technologies param"] == "GENSET_DIESEL", "f_max"] = 1e15

    df.loc[df["Technologies param"] == "STOVE_WOOD", "f_min"] = stove_wood_fmin[k]
    df.loc[df["Technologies param"] == "STOVE_LPG",  "f_min"] = stove_lpg_fmin[k]
    df.loc[df["Technologies param"] == "STOVE_ELEC", "f_min"] = 0.0

    df.loc[df["Technologies param"] == "PV_UTILITY", "f_min"] = PV_FMIN_GW[k]

    for tech in POPULATION_SCALED:
        mask = df["Technologies param"] == tech
        if mask.any():
            df.loc[mask, "f_min"] = base_fmin[tech] * share

    # REGASIFICATION and LNG_STORAGE f_min = 0:
    # AETN 2024 confirms no LNG regasification stations in Norte Amazónica
    # (Riberalta, Guayaramerín, Cobija all use diesel only for generation)
        df.loc[df["Technologies param"] == "REGASIFICATION", "f_min"] = 0.0  # Existing line
        df.loc[df["Technologies param"] == "LNG_STORAGE",    "f_min"] = 0.0  # Existing line

        # Override ST_SNG: set both f_min and f_max to 0 for all clusters (requested change)
        df.loc[df["Technologies param"] == "ST_SNG", "f_min"] = 0.0
        df.loc[df["Technologies param"] == "ST_SNG", "f_max"] = 0.0

    # LED-only lighting (see markdown above)
    df.loc[df["Technologies param"] == "LED_BULB",  "fmin_perc"] = 1.0
    df.loc[df["Technologies param"] == "LED_BULB",  "fmax_perc"] = 1.0
    df.loc[df["Technologies param"] == "LED_LIGHT", "fmin_perc"] = 1.0
    df.loc[df["Technologies param"] == "LED_LIGHT", "fmax_perc"] = 1.0
    df.loc[df["Technologies param"] == "CONVENTIONAL_BULB",  "f_max"] = 0.0
    df.loc[df["Technologies param"] == "CONVENTIONAL_LIGHT", "f_max"] = 0.0

    # Remove any pre-existing SHS rows, then append this cluster's own
    shs_df = pd.DataFrame(shs_rows_for(k), columns=base_df.columns)
    df = df[~df["Technologies param"].isin(shs_df["Technologies param"])]
    df = pd.concat([df, shs_df], ignore_index=True)

    out_path = os.path.join(OUT_DIR, f"C{k}", "Technologies.csv")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    df.to_csv(out_path, sep=";", index=False)
print(f"Saved Technologies.csv files")

Saved Technologies.csv files


## 6. Verification

In [7]:
clusters_out = {}
for k in range(1, 6):
    path = os.path.join(OUT_DIR, f"C{k}", "Technologies.csv")
    clusters_out[k] = pd.read_csv(path, sep=";")

def lookup(df, tech, col):
    row = df.loc[df["Technologies param"] == tech, col]
    return float(row.values[0]) if len(row) else float("nan")

checks = [
    ("GENSET_DIESEL",       "f_min"),
    ("STOVE_WOOD",          "f_min"),
    ("STOVE_LPG",           "f_min"),
    ("PV_UTILITY",          "f_min"),
    ("DIESEL_STORAGE",      "f_min"),
    ("LNG_STORAGE",         "f_min"),
    ("REGASIFICATION",      "f_min"),
    ("CONVENTIONAL_BULB",   "f_max"),
    ("CONVENTIONAL_LIGHT",  "f_max"),
]

header = f"{'Technology':<30}" + "".join(f"  C{k:>9}" for k in range(1, 6))
print(header)
print("-" * len(header))
for tech, col in checks:
    vals = [lookup(clusters_out[k], tech, col) for k in range(1, 6)]
    print(f"{tech+' '+col:<30}" + "".join(f"  {v:>9.5f}" for v in vals))

print()
for shs in ["PV_HS", "HS_DIESEL", "BATT_HS"]:
    present = ["✓" if shs in clusters_out[k]["Technologies param"].values else "✗" for k in range(1, 6)]
    print(f"{shs+' present':<30}" + "".join(f"  {p:>9}" for p in present))

Technology                      C        1  C        2  C        3  C        4  C        5
------------------------------------------------------------------------------------------
GENSET_DIESEL f_min               0.00000    0.00000    0.05333    0.00742    0.02736
STOVE_WOOD f_min                  0.00330    0.00022    0.00462    0.00380    0.00020
STOVE_LPG f_min                   0.01530    0.00123    0.08229    0.02854    0.03736
PV_UTILITY f_min                  0.00000    0.00000    0.00000    0.00000    0.00510
DIESEL_STORAGE f_min              3.91205    0.38365   19.27704    7.53516    7.44523
LNG_STORAGE f_min                 0.00000    0.00000    0.00000    0.00000    0.00000
REGASIFICATION f_min              0.00000    0.00000    0.00000    0.00000    0.00000
CONVENTIONAL_BULB f_max           0.00000    0.00000    0.00000    0.00000    0.00000
CONVENTIONAL_LIGHT f_max          0.00000    0.00000    0.00000    0.00000    0.00000

PV_HS present                           ✓  